In [1]:
import pandas as pd

In [2]:
read_dir = '/Users/ramprasad.sa/hallucinations_annotations_trial_round2/data_new'
filename = 'llm_as_judge_batch.csv'
df = pd.read_csv(f'{read_dir}/{filename}')
df.head()

,Unnamed: 0,user_id,span_uuid,docid,source,summary,model,benchmark_dataset_name,origin,nonfactual_span,error_type,inference_likelihood,inference_knowledge,inference_consequence,eval_id,eval_nonfactual_span,eval_error_type,eval_inference_likelihood,eval_inference_knowledge,eval_inference_consequence
0,0,evaluator,372b3fa72cf04b4ba2006ae3c081ab2a,ACIBENCH-D2N120:gpt4-ul2,[doctor] hello larry how are you doing today [...,"Larry, the patient, visited his doctor followi...",gpt4,genaudit,ACIBENCH,unable to lead an active lifestyle,inference,likely,common,low_consequence,ann_2,unable to lead an active lifestyle,inference,likely,common,low_consequence
1,1,evaluator,372b3fa72cf04b4ba2006ae3c081ab2a,ACIBENCH-D2N120:gpt4-ul2,[doctor] hello larry how are you doing today [...,"Larry, the patient, visited his doctor followi...",gpt4,genaudit,ACIBENCH,unable to lead an active lifestyle,inference,likely,common,low_consequence,llama3_8b,unable to lead an active lifestyle,somewhat_mistake,likely,common,moderate_consequence
2,2,evaluator,372b3fa72cf04b4ba2006ae3c081ab2a,ACIBENCH-D2N120:gpt4-ul2,[doctor] hello larry how are you doing today [...,"Larry, the patient, visited his doctor followi...",gpt4,genaudit,ACIBENCH,unable to lead an active lifestyle,inference,likely,common,low_consequence,qwen2.5_7b,unable to lead an active lifestyle,inference,likely,common,low_consequence
3,3,evaluator,372b3fa72cf04b4ba2006ae3c081ab2a,ACIBENCH-D2N120:gpt4-ul2,[doctor] hello larry how are you doing today [...,"Larry, the patient, visited his doctor followi...",gpt4,genaudit,ACIBENCH,unable to lead an active lifestyle,inference,likely,common,low_consequence,qwen2.5_72b,unable to lead an active lifestyle,inference,somewhat_likely,common,low_consequence
4,4,evaluator,372b3fa72cf04b4ba2006ae3c081ab2a,ACIBENCH-D2N120:gpt4-ul2,[doctor] hello larry how are you doing today [...,"Larry, the patient, visited his doctor followi...",gpt4,genaudit,ACIBENCH,unable to lead an active lifestyle,inference,likely,common,low_consequence,gpt,unable to lead an active lifestyle,inference,likely,common,moderate_consequence


In [3]:
import psycopg2
conn = psycopg2.connect(
   database="d8i8l39r762feh", 
    user='ufnbklscc76qho', 
    password='paa07c080a16d99fe343e98dc65914f8a01384355312fcf7cb0135029f6fd2c49', 
    host='c229h9evb9laqg.cluster-czrs8kj4isg7.us-east-1.rds.amazonaws.com', 
    port = '5432'
    
)
cursor = conn.cursor()

In [3]:
len(df)

4241

In [4]:
import uuid
if 'uuid' not in df.keys():
    print('ADDING UUID...')
    df['uuid'] = [uuid.uuid4().hex for idx, row in df.iterrows()]
    # df.to_csv(f'{read_dir}/{filename}')

ADDING UUID...


In [5]:
df.head()[:1]

,Unnamed: 0,user_id,span_uuid,docid,source,summary,model,benchmark_dataset_name,origin,nonfactual_span,...,inference_likelihood,inference_knowledge,inference_consequence,eval_id,eval_nonfactual_span,eval_error_type,eval_inference_likelihood,eval_inference_knowledge,eval_inference_consequence,uuid
0,0,evaluator,372b3fa72cf04b4ba2006ae3c081ab2a,ACIBENCH-D2N120:gpt4-ul2,[doctor] hello larry how are you doing today [...,"Larry, the patient, visited his doctor followi...",gpt4,genaudit,ACIBENCH,unable to lead an active lifestyle,...,likely,common,low_consequence,ann_2,unable to lead an active lifestyle,inference,likely,common,low_consequence,6ea602bb3feb429f9ff42967bfab0f9d


In [6]:
df[df['uuid'] == '4d22961d61a249e4a653d3e214347f57']

,Unnamed: 0,user_id,span_uuid,docid,source,summary,model,benchmark_dataset_name,origin,nonfactual_span,...,inference_likelihood,inference_knowledge,inference_consequence,eval_id,eval_nonfactual_span,eval_error_type,eval_inference_likelihood,eval_inference_knowledge,eval_inference_consequence,uuid


In [9]:


sql_drop = '''DROP TABLE IF EXISTS model_summaries;'''

sql = '''CREATE TABLE model_summaries (
    uuid uuid DEFAULT gen_random_uuid() PRIMARY KEY, 
    docid TEXT NOT NULL ,
    source TEXT,
    summary TEXT NOT NULL,
    model TEXT NOT NULL, 
    benchmark_dataset_name TEXT,
    origin TEXT
);'''

#Creating a database
cursor.execute(sql_drop)
cursor.execute(sql)
print("Table created successfully........")
conn.commit()

Table created successfully........


In [10]:
df.keys()

Index(['Unnamed: 0', 'user_id', 'span_uuid', 'docid', 'source', 'summary',
       'model', 'benchmark_dataset_name', 'origin', 'nonfactual_span',
       'error_type', 'inference_likelihood', 'inference_knowledge',
       'inference_consequence', 'eval_id', 'eval_nonfactual_span',
       'eval_error_type', 'eval_inference_likelihood',
       'eval_inference_knowledge', 'eval_inference_consequence', 'uuid'],
      dtype='object')

In [11]:
len(set(df['docid'].values))

208

In [12]:
import re

df_source_summaries = df.drop_duplicates(subset='docid', keep="last")

for idx, row in df_source_summaries.iterrows():
    # print(row)
    uuid = row['uuid']
    span_uuid = row['span_uuid']
    docid = row['docid']
    source = row['source']
    summary = row['summary']
    model = row['model']
    benchmark_dataset_name = row['benchmark_dataset_name']
    origin = row['origin']
    summary = re.sub( "'", r"''", summary)
    source = re.sub( "'", r"''", source)
    # value_str = f"""'{uuid}', '{docid}', '{source}', '{summary}', '{model}', '{benchmark_dataset_name}', '{origin}'"""
        # print(re.findall(r"\'(\w+)\'", dialogue))
    # print(value_str)
    cursor.execute(
    """
    INSERT INTO model_summaries (
        uuid, docid, source, summary, model,
        benchmark_dataset_name, origin
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    """,
    (
        uuid,
        docid,
        source,
        summary,
        model,
        benchmark_dataset_name,
        origin,
    )
    )

    # cursor.execute(f"""INSERT INTO modified_label(uuid, user_id, docid, source, summary, benchmark_dataset_name, origin, nonfactual_span, error_type, mistake_severity, inference_likelihood, inference_knowledge, inference_severity, attempt_number) VALUES ({value_str})""")

In [13]:
cursor.execute("""SELECT * from model_summaries""")

print('Size after adding' , len(cursor.fetchall()))

Size after adding 208


In [14]:
cursor = conn.cursor()
sql_drop = '''DROP TABLE IF EXISTS label_evaluators;'''



create_table_label_str = '''CREATE TABLE label_evaluators (
    uuid TEXT,
    span_uuid TEXT,
    user_id TEXT NOT NULL,
    docid TEXT NOT NULL,
    source TEXT NOT NULL,
    summary TEXT NOT NULL,
    model TEXT NOT NULL,
    benchmark_dataset_name TEXT,
    origin TEXT,
    nonfactual_span TEXT,
    error_type TEXT,
    inference_likelihood TEXT,
    inference_knowledge TEXT,
    inference_consequence TEXT
);'''

cursor.execute(sql_drop)
cursor.execute(create_table_label_str)
print("Table created successfully........")
conn.commit()

Table created successfully........


In [15]:
df.keys()

Index(['Unnamed: 0', 'user_id', 'span_uuid', 'docid', 'source', 'summary',
       'model', 'benchmark_dataset_name', 'origin', 'nonfactual_span',
       'error_type', 'inference_likelihood', 'inference_knowledge',
       'inference_consequence', 'eval_id', 'eval_nonfactual_span',
       'eval_error_type', 'eval_inference_likelihood',
       'eval_inference_knowledge', 'eval_inference_consequence', 'uuid'],
      dtype='object')

In [16]:
# df.drop_duplicates(subset='span_uuid', keep="last")

In [17]:
import re

df_spans_distinct = df.drop_duplicates(subset='span_uuid', keep="first")

for idx, row in df_spans_distinct.iterrows():
    # print(row)
    uuid = row['uuid']
    span_uuid = row['span_uuid']
    user_id = row['user_id']
    docid = row['docid']
    source = row['source']
    summary = row['summary']
    model = row['model']
    benchmark_dataset_name = row['benchmark_dataset_name']
    origin = row['origin']
    nonfactual_span = row['nonfactual_span']
    error_type = row['error_type']
    inference_likelihood = row['inference_likelihood']
    inference_knowledge = row['inference_knowledge']
    inference_consequence = row['inference_consequence']
    summary = re.sub( "'", r"''", summary)
    source = re.sub( "'", r"''", source)
    value_str = f"""'{uuid}', '{user_id}', '{docid}', '{source}', '{summary}', '{model}', '{benchmark_dataset_name}', '{origin}',
    '{nonfactual_span}', '{error_type}', '{inference_likelihood}', '{inference_knowledge}', '{inference_consequence}'"""
        # print(re.findall(r"\'(\w+)\'", dialogue))
    # print(value_str)
    cursor.execute(
    """
    INSERT INTO label_evaluators (
        uuid,
        span_uuid, 
        user_id, 
        docid, 
        source, 
        summary, 
        model,
        benchmark_dataset_name, 
        origin,
        nonfactual_span,
        error_type,
        inference_likelihood, 
        inference_knowledge, 
        inference_consequence
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """,
    (
        uuid,
        span_uuid,
        user_id,
        docid,
        source,
        summary,
        model,
        benchmark_dataset_name,
        origin, 
        nonfactual_span.strip(), 
        error_type,
        inference_likelihood, 
        inference_knowledge, 
        inference_consequence
    )
    )

    # cursor.execute(f"""INSERT INTO modified_label(uuid, user_id, docid, source, summary, benchmark_dataset_name, origin, nonfactual_span, error_type, mistake_severity, inference_likelihood, inference_knowledge, inference_severity, attempt_number) VALUES ({value_str})""")

In [18]:
cursor.execute("""SELECT * from label_evaluators""")

print('Size after adding' , len(cursor.fetchall()))

Size after adding 1531


In [19]:
cursor = conn.cursor()
sql_drop = '''DROP TABLE IF EXISTS label_evaluators_validated;'''
# d SERIAL PRIMARY KEY,
#     source_id INT NOT NULL,
#     start_pos INT NOT NULL,
#     end_pos INT NOT NULL,
#     text TEXT NOT NULL,
#     category VARCHAR(50) NOT NULL,
#     created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP


create_table_label_str = '''CREATE TABLE label_evaluators_validated (
    uuid TEXT, 
    span_uuid TEXT,
    user_id TEXT NOT NULL,
    docid TEXT NOT NULL,
    source TEXT NOT NULL,
    summary TEXT NOT NULL,
    model TEXT NOT NULL,
    benchmark_dataset_name TEXT,
    origin TEXT,
    nonfactual_span TEXT,
    error_type TEXT,
    inference_likelihood TEXT,
    inference_knowledge TEXT,
    inference_consequence TEXT,
    attempt_number INT
);'''

cursor.execute(sql_drop)
cursor.execute(create_table_label_str)
print("Table created successfully........")
conn.commit()

Table created successfully........


In [9]:
cursor.execute("""SELECT * from label_evaluators_validated""")

print('Size after adding' , len(cursor.fetchall()))

Size after adding 13


In [10]:
cursor.execute("""SELECT * from label_evaluators_validated""")
cursor.fetchall()

[(None,
  '5fc5f1c594514142bd3172226aabd3ee',
  'sanjana',
  'ACIBENCH-D2N120:gpt4-ul2',
  "[doctor] hello larry how are you doing today [patient] i''ve been better my primary care doctor wanted me to see you for this back pain that i''ve been having for a couple of years now [doctor] okay [patient] and i have tried so many things and just nothing seems to work [doctor] i''m sorry to hear that now do you remember what caused the pain initially [patient] you know i never fell or was in an accident but i do remember it all happened like i started having back pain after i was moving furniture one weekend [doctor] okay and now can you describe the pain for me [patient] yeah it feels like a sharp stabbing pain in my back and it does go down even down to my right leg sometimes all the way down to my toe it feels like my big toe [doctor] okay [patient] and i also feel like i just ca n''t get comfortable which is worse when i sit down okay yeah [doctor] okay now with that do you have any numbn

In [5]:
cursor.fetchall()

[]